# Testes

### Configuração de ambiente

In [ ]:
from os import environ

environ['CUDA_VISIBLE_DEVICES'] = input('GPU ID: ')

### Imports

In [ ]:
from os.path import join
from os import makedirs
from json import load, dump
from datetime import datetime

from unsloth import FastVisionModel
from tqdm.notebook import tqdm
from PIL import Image

import torch

from scripts.authentication import authenticate_huggingface
from scripts.messages import add_inference_message, format_prompt
from scripts.data import SimpleLesionData, SimpleDatasetAnalysis
from scripts.test import Test, TestResult, GenerationParameters

import scripts.definitions as defs

### Autenticação

In [ ]:
authenticate_huggingface()

### Configurações

In [ ]:
MODEL = defs.BASE_MODEL_NAME
QUANTIZED = False  # Isso é sobreescrito no caso de modelos treinados
TEST_ON_TEST = True  # Testa o modelo sobre os dados de teste
TEST_ON_TRAINING = False  # Testa o modelo sobre os dados de treinamento
SAVE_FREQUENCY = 10  # Salva os resultados a cada N iterações
TEMPERATURE = 0.005
BATCH_SIZE = 8

with open(join(defs.TRAINING_PATH, 'models.json'), 'r', encoding='utf-8') as file:
    models = {model_name: defs.Model(**model) for model_name, model in load(file).items()}

model_stats = models[MODEL]
model_path = ''

if model_stats.local:
    model_path = join(defs.RESULTS_PATH, 'adapter_weights', MODEL)
else:
    model_path = MODEL

quantized = model_stats.quantized if model_stats.quantized is not None else QUANTIZED
prompt_type = model_stats.prompt_type
model_version = model_stats.version
model_size = model_stats.size

### Carregamento do dataset

In [ ]:
with open(join(defs.DATA_PATH, 'stt_data', 'training_dataset.json'), 'r', encoding='utf-8') as file:
    training_dataset = [SimpleLesionData(**data) for data in load(file)]

with open(join(defs.DATA_PATH, 'training_dataset_analysis.json'), 'r', encoding='utf-8') as file:
    training_dataset_analysis = SimpleDatasetAnalysis(**load(file))

with open(join(defs.DATA_PATH, 'stt_data', 'test_dataset.json'), 'r', encoding='utf-8') as file:
    test_dataset = [SimpleLesionData(**data) for data in load(file)]

with open(join(defs.DATA_PATH, 'test_dataset_analysis.json'), 'r', encoding='utf-8') as file:
    test_dataset_analysis = SimpleDatasetAnalysis(**load(file))

### Carregamento do modelo

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_path,
    load_in_4bit=quantized,
    use_gradient_checkpointing='unsloth',
    device_map={'': 0}
)

FastVisionModel.for_inference(model)

### Preparação do teste

In [ ]:
formatted_prompt = format_prompt(prompt_type, training_dataset_analysis)
messages = add_inference_message(formatted_prompt)

test_name = f'{MODEL}_test_{datetime.now().isoformat()}'.strip('unsloth/')

test = Test(
    tested_model=MODEL.replace('unsloth/', ''),
    model=model_stats,
    generation_parameters=GenerationParameters(
        max_new_tokens=512,
        temperature=TEMPERATURE
    ),
    results_on_test_data=[],
    results_on_training_data=[],
)

tests_path = join(defs.RESULTS_PATH, 'tests')

makedirs(tests_path, exist_ok=True)
test_path = join(tests_path, f'{test_name}.json')

### Testes sobre os dados de teste

In [ ]:
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

if TEST_ON_TEST:
    for i in tqdm(range(0, len(test_dataset), BATCH_SIZE), desc='Testando com dados de teste: '):
        batch = test_dataset[i:i + BATCH_SIZE]
        images = [[Image.open(join(defs.DATA_PATH, 'stt_data', 'images', d.image)).convert('RGB')] for d in batch]

        inputs = tokenizer(
            images,
            [input_text] * len(batch),
            add_special_tokens=False,
            return_tensors='pt',
            padding=True,
        ).to('cuda')

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=TEMPERATURE,
        )

        for lesion_data, output in zip(batch, outputs):
            decoded = tokenizer.decode(output, skip_special_tokens=True)
            assistant_message = decoded.split('assistant')[-1].strip()
            result = TestResult(
                exam_id=lesion_data.exam_id,
                image=lesion_data.image,
                answer=assistant_message
            )
            test.results_on_test_data.append(result)

        if (i // BATCH_SIZE + 1) % SAVE_FREQUENCY == 0:
            with open(test_path, 'w+', encoding='utf-8') as file:
                dump(test.model_dump(), file, indent=4, ensure_ascii=False)

### Testes sobre os dados de treinamento

In [ ]:
if TEST_ON_TRAINING:
    for i in tqdm(range(0, len(training_dataset), BATCH_SIZE), desc='Testando com dados de treinamento: '):
        batch = training_dataset[i:i + BATCH_SIZE]
        images = [[Image.open(join(defs.DATA_PATH, 'stt_data', 'images', d.image)).convert('RGB')] for d in batch]

        inputs = tokenizer(
            images,
            [input_text] * len(batch),
            add_special_tokens=False,
            return_tensors='pt',
            padding=True,
        ).to('cuda')

        outputs = model.generate(
            **inputs,
            max_new_tokens=defs.MAX_TOKENS,
            temperature=TEMPERATURE,
        )

        for lesion_data, output in zip(batch, outputs):
            decoded = tokenizer.decode(output, skip_special_tokens=True)
            assistant_message = decoded.split('assistant')[-1].strip()
            result = TestResult(
                exam_id=lesion_data.exam_id,
                image=lesion_data.image,
                answer=assistant_message
            )
            test.results_on_training_data.append(result)

        if (i // BATCH_SIZE + 1) % SAVE_FREQUENCY == 0:
            with open(test_path, 'w+', encoding='utf-8') as file:
                dump(test.model_dump(), file, indent=4, ensure_ascii=False)

### Salvamento do teste

In [ ]:
with open(test_path, 'w+', encoding='utf-8') as file:
    dump(test.model_dump(), file, indent=4, ensure_ascii=False)